# YOLO11 Object Detection Training Pipeline (iPhone)

### 1. Import Core Utilities
Imports the Python modules used across data checks and reporting.

In [1]:
from collections import Counter
from pathlib import Path
import random
import shutil
import time

import albumentations as A
import cv2
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

### 2. Define Classes and Locate Data Root
Sets class IDs, image extensions, and auto-detects the `Data/Iphone` folder from the current working path.

In [2]:
CLASS_NAME_TO_ID = {
    "blue bins": 0,
    "metal can": 1,
    "newspaper": 2,
    "plastic bag": 3,
    "plastic bottle": 4,
}

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

def find_iphone_data_root(start: Path) -> Path:
    """Find the Data/Iphone folder by walking up from the notebook working directory."""
    start = start.resolve()
    for base in [start, *start.parents]:
        candidate = base / "Data" / "Iphone"
        if candidate.exists() and candidate.is_dir():
            return candidate
    raise FileNotFoundError("Could not find Data/Iphone from current working directory.")

DATA_ROOT = find_iphone_data_root(Path.cwd())
print(f"Using data root: {DATA_ROOT}")

Using data root: C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Iphone


### 3. Audit Images and Label Quality
Counts images, label files, and object annotations per source folder, then prints per-class and total summaries.

In [9]:
def count_classes_in_label_dir(label_dir: Path) -> Counter:
    class_counter = Counter()
    for txt_path in label_dir.glob("*.txt"):
        text = txt_path.read_text(encoding="utf-8").strip()
        if not text:
            continue
        for line in text.splitlines():
            parts = line.split()
            # YOLO row format: class_id x_center y_center width height
            if len(parts) < 5:
                continue
            try:
                class_id = int(parts[0])
                float(parts[1]); float(parts[2]); float(parts[3]); float(parts[4])
            except ValueError:
                continue
            class_counter[class_id] += 1
    return class_counter

label_dirs = [
    DATA_ROOT / "Sam3 Metal can output" / "bbox",
    DATA_ROOT / "Sam3 Newspaper output" / "bbox",
    DATA_ROOT / "Sam3 Plastic bag output" / "bbox",
    DATA_ROOT / "Sam3 Plastic bottle output" / "bbox",
]

overall_counter = Counter()
for label_dir in label_dirs:
    if label_dir.exists() and label_dir.is_dir():
        overall_counter.update(count_classes_in_label_dir(label_dir))

id_to_name = {v: k for k, v in CLASS_NAME_TO_ID.items()}
for class_id in sorted(id_to_name):
    print(f"{id_to_name[class_id]}: {overall_counter.get(class_id, 0)}")

blue bins: 460
metal can: 119
newspaper: 101
plastic bag: 120
plastic bottle: 120


### 4. Build a Unified YOLO Dataset Folder
Copies valid image-label pairs into a clean `yolo_iphone_dataset` structure for downstream processing.

In [ ]:
DATASET_ROOT = DATA_ROOT / "yolo_iphone_dataset"
IMAGES_OUT = DATASET_ROOT / "images"
LABELS_OUT = DATASET_ROOT / "labels"
IMAGES_OUT.mkdir(parents=True, exist_ok=True)
LABELS_OUT.mkdir(parents=True, exist_ok=True)

# Start clean so reruns do not duplicate files.
for p in IMAGES_OUT.glob("*"):
    if p.is_file():
        p.unlink()
for p in LABELS_OUT.glob("*"):
    if p.is_file():
        p.unlink()

SOURCE_CLASS_NAMES = ["metal can", "newspaper", "plastic bag", "plastic bottle"]

def next_available_path(path: Path) -> Path:
    if not path.exists():
        return path
    stem = path.stem
    suffix = path.suffix
    i = 1
    while True:
        candidate = path.with_name(f"{stem}_{i}{suffix}")
        if not candidate.exists():
            return candidate
        i += 1

rows = []
for class_name in SOURCE_CLASS_NAMES:
    src_img_dir = DATA_ROOT / class_name
    src_lbl_dir = DATA_ROOT / f"Sam3 {class_name} output" / "bbox"

    if not src_img_dir.exists() or not src_lbl_dir.exists():
        rows.append({
            "class_name": class_name,
            "available_pairs": 0,
            "copied_pairs": 0,
            "status": "missing source folder",
        })
        continue

    images = sorted([p for p in src_img_dir.glob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES])
    paired = []
    for img_path in images:
        lbl_path = src_lbl_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            paired.append((img_path, lbl_path))

    copied = 0
    for img_path, lbl_path in paired:
        dst_img = next_available_path(IMAGES_OUT / img_path.name)
        dst_lbl = next_available_path(LABELS_OUT / lbl_path.name)
        shutil.copy2(str(img_path), str(dst_img))
        shutil.copy2(str(lbl_path), str(dst_lbl))
        copied += 1

    rows.append({
        "class_name": class_name,
        "available_pairs": len(paired),
        "copied_pairs": copied,
        "status": "ok",
    })

copy_df = pd.DataFrame(rows)
display(copy_df)

print(f"Dataset root: {DATASET_ROOT}")
print(f"Images folder: {IMAGES_OUT}")
print(f"Labels folder: {LABELS_OUT}")
print(f"Total copied pairs: {int(copy_df['copied_pairs'].sum())}")

,class_name,available_pairs,copied_pairs,status
0,metal can,119,119,ok
1,newspaper,101,101,ok
2,plastic bag,120,120,ok
3,plastic bottle,120,120,ok


Dataset root: C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Iphone\yolo_iphone_dataset
Images folder: C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Iphone\yolo_iphone_dataset\images
Labels folder: C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Iphone\yolo_iphone_dataset\labels
Total copied pairs: 460


### 5. Balance Recyclable Classes with Augmentation
Uses Albumentations to generate additional samples so each recyclable class moves toward the target count.

In [ ]:
TARGET_PER_CLASS = 300
random.seed(42)

images_dir = DATASET_ROOT / "images"
labels_dir = DATASET_ROOT / "labels"
valid_ext = {".jpg", ".jpeg", ".png"}

# Blue bins is intentionally excluded from balancing target classes.
BLUE_ID = CLASS_NAME_TO_ID["blue bins"]
TARGET_CLASS_NAMES = ["metal can", "newspaper", "plastic bag", "plastic bottle"]
TARGET_CLASS_IDS = [CLASS_NAME_TO_ID[name] for name in TARGET_CLASS_NAMES]
id_to_name = {v: k for k, v in CLASS_NAME_TO_ID.items()}

if not images_dir.exists() or not labels_dir.exists():
    raise FileNotFoundError("Dataset folders not found. Run the copy dataset cell first.")

def sanitize_yolo_bbox(bbox):
    x, y, w, h = bbox
    w = min(max(w, 1e-6), 1.0)
    h = min(max(h, 1e-6), 1.0)
    half_w = w / 2.0
    half_h = h / 2.0
    x = min(max(x, half_w), 1.0 - half_w)
    y = min(max(y, half_h), 1.0 - half_h)
    return [x, y, w, h]

def read_yolo_entries(label_path: Path):
    text = label_path.read_text(encoding="utf-8").strip()
    if not text:
        return []
    entries = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            class_id = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
        except ValueError:
            continue
        entries.append((class_id, sanitize_yolo_bbox([x, y, w, h])))
    return entries

def write_yolo_entries(label_path: Path, entries):
    lines = []
    for class_id, bbox in entries:
        x, y, w, h = sanitize_yolo_bbox(bbox)
        lines.append(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

def find_image_for_stem(stem: str):
    for ext in valid_ext:
        img = images_dir / f"{stem}{ext}"
        if img.exists():
            return img
    return None

def infer_recyclable_class_id(entries):
    recyclable_ids = sorted({cid for cid, _ in entries if cid in TARGET_CLASS_IDS})
    if len(recyclable_ids) == 1:
        return recyclable_ids[0]
    return None

def next_aug_paths(class_id: int, seq_num: int):
    while True:
        stem = f"aug_bal_c{class_id}_{seq_num:04d}"
        out_img = images_dir / f"{stem}.jpg"
        out_lbl = labels_dir / f"{stem}.txt"
        if not out_img.exists() and not out_lbl.exists():
            return stem, out_img, out_lbl, seq_num
        seq_num += 1

samples_by_class = {cid: [] for cid in TARGET_CLASS_IDS}

for lbl_path in labels_dir.glob("*.txt"):
    entries = read_yolo_entries(lbl_path)
    if not entries:
        continue
    rec_id = infer_recyclable_class_id(entries)
    if rec_id is None:
        continue

    img_path = find_image_for_stem(lbl_path.stem)
    if img_path is None:
        continue

    samples_by_class[rec_id].append((img_path, lbl_path, entries))

before_rows = []
for cid in TARGET_CLASS_IDS:
    before_rows.append({
        "class_id": cid,
        "class_name": id_to_name[cid],
        "count_before": len(samples_by_class.get(cid, [])),
    })
before_df = pd.DataFrame(before_rows).sort_values("class_id").reset_index(drop=True)
display(before_df)

augment_pipeline = A.Compose(
    [
        A.Affine(
            scale=(1.0, 1.0),
            rotate=(0, 0),
            shear=(0, 0),
            translate_percent={"x": (-0.04, 0.04), "y": (-0.04, 0.04)},
            p=1.0,
        ),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.0, p=0.5),
    ],
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.2),
)

created_rows = []
for class_id in TARGET_CLASS_IDS:
    class_samples = samples_by_class.get(class_id, [])
    current_count = len(class_samples)
    needed = max(TARGET_PER_CLASS - current_count, 0)

    if current_count == 0 and needed > 0:
        created_rows.append({
            "class_id": class_id,
            "class_name": id_to_name[class_id],
            "needed": needed,
            "created": 0,
            "final_estimate": 0,
            "status": "no source samples",
        })
        continue

    created = 0
    attempts = 0
    max_attempts = max(needed * 80, 200) if needed > 0 else 0
    seq_num = current_count + 1

    while created < needed and attempts < max_attempts:
        attempts += 1

        src_img_path, _, src_entries = random.choice(class_samples)
        img = cv2.imread(str(src_img_path))
        if img is None:
            continue

        class_labels = [cid for cid, _ in src_entries]
        bboxes = [bbox for _, bbox in src_entries]

        try:
            transformed = augment_pipeline(image=img, bboxes=bboxes, class_labels=class_labels)
        except Exception:
            continue

        if len(transformed["bboxes"]) == 0:
            continue

        new_entries = []
        for cid, bb in zip(transformed["class_labels"], transformed["bboxes"]):
            new_entries.append((int(cid), sanitize_yolo_bbox(bb)))

        # Keep only samples where the target recyclable class still exists.
        if class_id not in [cid for cid, _ in new_entries]:
            continue

        _, out_img_path, out_lbl_path, seq_num = next_aug_paths(class_id, seq_num)
        seq_num += 1

        if cv2.imwrite(str(out_img_path), transformed["image"]):
            write_yolo_entries(out_lbl_path, new_entries)
            created += 1

    created_rows.append({
        "class_id": class_id,
        "class_name": id_to_name[class_id],
        "needed": needed,
        "created": created,
        "final_estimate": current_count + created,
        "status": "ok" if (current_count + created) >= TARGET_PER_CLASS else "below target",
    })

created_df = pd.DataFrame(created_rows).sort_values("class_id").reset_index(drop=True)
display(created_df)

# Recount recyclable-class images after augmentation.
final_rows = []
for cid in TARGET_CLASS_IDS:
    count_after = 0
    for lbl_path in labels_dir.glob("*.txt"):
        entries = read_yolo_entries(lbl_path)
        if not entries:
            continue
        rec_id = infer_recyclable_class_id(entries)
        if rec_id == cid:
            img_path = find_image_for_stem(lbl_path.stem)
            if img_path is not None:
                count_after += 1
    final_rows.append({
        "class_id": cid,
        "class_name": id_to_name[cid],
        "count_after": count_after,
    })

final_df = pd.DataFrame(final_rows).sort_values("class_id").reset_index(drop=True)
display(final_df)

print("\nBalancing complete (recyclables only).")
print(f"Target per recyclable class: {TARGET_PER_CLASS}")
print("Blue bins were not used as a balancing target class.")

,class_id,class_name,count_before
0,1,metal can,119
1,2,newspaper,101
2,3,plastic bag,120
3,4,plastic bottle,120


,class_id,class_name,needed,created,final_estimate,status
0,1,metal can,181,181,300,ok
1,2,newspaper,199,199,300,ok
2,3,plastic bag,180,180,300,ok
3,4,plastic bottle,180,180,300,ok


,class_id,class_name,count_after
0,1,metal can,300
1,2,newspaper,300
2,3,plastic bag,300
3,4,plastic bottle,300



Balancing complete (recyclables only).
Target per recyclable class: 300
Blue bins were not used as a balancing target class.


### 6. Split Dataset and Train YOLO11
Creates train/val/test splits, writes dataset YAML, and trains a YOLO11 model on GPU with epoch timing callbacks.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Please select a GPU-enabled runtime.")

random.seed(42)
valid_ext = {".jpg", ".jpeg", ".png"}

# Build pairs directly from existing checked labels/images in yolo_iphone_dataset.
paired = []
for lbl_path in (DATASET_ROOT / "labels").glob("*.txt"):
    img_path = None
    for ext in valid_ext:
        candidate = (DATASET_ROOT / "images") / f"{lbl_path.stem}{ext}"
        if candidate.exists():
            img_path = candidate
            break
    if img_path is not None:
        paired.append((img_path, lbl_path))

if len(paired) == 0:
    raise RuntimeError("No paired image/label samples found in yolo_iphone_dataset.")

# Build train/val/test split
SPLIT_ROOT = DATASET_ROOT / "split"
for split_name in ["train", "val", "test"]:
    (SPLIT_ROOT / "images" / split_name).mkdir(parents=True, exist_ok=True)
    (SPLIT_ROOT / "labels" / split_name).mkdir(parents=True, exist_ok=True)

# Clear old split files for deterministic reruns
for p in (SPLIT_ROOT / "images").rglob("*"):
    if p.is_file():
        p.unlink()
for p in (SPLIT_ROOT / "labels").rglob("*"):
    if p.is_file():
        p.unlink()

random.shuffle(paired)
n = len(paired)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
n_test = n - n_train - n_val
splits = {
    "train": paired[:n_train],
    "val": paired[n_train:n_train + n_val],
    "test": paired[n_train + n_val:],
}

for split_name, items in splits.items():
    for img_path, lbl_path in items:
        dst_img = SPLIT_ROOT / "images" / split_name / img_path.name
        dst_lbl = SPLIT_ROOT / "labels" / split_name / lbl_path.name
        shutil.copy2(str(img_path), str(dst_img))
        shutil.copy2(str(lbl_path), str(dst_lbl))

class_names = [name for name, _ in sorted(CLASS_NAME_TO_ID.items(), key=lambda kv: kv[1])]
dataset_yaml = {
    "path": str(SPLIT_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": class_names,
}

yaml_path = DATASET_ROOT / "iphone_dataset.yaml"
with yaml_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(dataset_yaml, f, sort_keys=False, allow_unicode=False)

print(f"Dataset YAML: {yaml_path}")
print(f"Total pairs: {n}")
print(f"Train/Val/Test: {len(splits['train'])}/{len(splits['val'])}/{len(splits['test'])}")

# Train YOLO11n on GPU only and print epoch duration.
model = YOLO("yolo11n.pt")
_epoch_timer = {"t0": None}

def on_train_epoch_start(trainer):
    _epoch_timer["t0"] = time.perf_counter()

def on_train_epoch_end(trainer):
    t0 = _epoch_timer.get("t0")
    if t0 is None:
        return
    dt = time.perf_counter() - t0
    print(f"[Epoch {trainer.epoch + 1}/{trainer.epochs}] duration: {dt:.2f}s")

model.add_callback("on_train_epoch_start", on_train_epoch_start)
model.add_callback("on_train_epoch_end", on_train_epoch_end)

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    workers=2,
    cache=False,
    amp=True,
    project=str(DATASET_ROOT / "runs"),
    name="yolo11n_iphone",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=42,
    device=0,
    verbose=True,
)

print("\nTraining complete.")
print(f"Best checkpoint: {Path(results.save_dir) / 'weights' / 'best.pt'}")
print(f"Run directory: {results.save_dir}")

### 7. Summarize Class Counts in Unified Labels Folder
Reads all label files in `yolo_iphone_dataset/labels` and prints total counts per class.

In [10]:
LABEL_DIR = Path(r"C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Iphone\yolo_iphone_dataset\labels")

def count_classes_in_label_dir(label_dir: Path) -> Counter:
    class_counter = Counter()
    for txt_path in label_dir.glob("*.txt"):
        text = txt_path.read_text(encoding="utf-8").strip()
        if not text:
            continue
        for line in text.splitlines():
            parts = line.split()
            # YOLO row format: class_id x_center y_center width height
            if len(parts) < 5:
                continue
            try:
                class_id = int(parts[0])
                float(parts[1]); float(parts[2]); float(parts[3]); float(parts[4])
            except ValueError:
                continue
            class_counter[class_id] += 1
    return class_counter

overall_counter = count_classes_in_label_dir(LABEL_DIR) if LABEL_DIR.exists() else Counter()
id_to_name = {v: k for k, v in CLASS_NAME_TO_ID.items()}

for class_id in sorted(id_to_name):
    print(f"{id_to_name[class_id]}: {overall_counter.get(class_id, 0)}")

blue bins: 1200
metal can: 300
newspaper: 300
plastic bag: 300
plastic bottle: 300


### 8. Run Validation Inference and Save Overlays
Loads the trained weights, predicts on validation images, and writes visualized bounding-box overlays.

In [ ]:
weights_path = Path(r"C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Code\yolo11l_iphone\weights\best.pt")
val_dir = Path(r"C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Iphone\yolo_iphone_dataset\split\images\val")
overlay_dir = val_dir.parent / "val_overlay"
overlay_dir.mkdir(parents=True, exist_ok=True)

# Load model and run prediction on validation images
model = YOLO(str(weights_path))
results = model.predict(source=str(val_dir), conf=0.25, iou=0.7, save=False, verbose=False)

# Save per-image overlays with bounding boxes/labels
saved = 0
for res in results:
    plotted = res.plot()
    out_path = overlay_dir / Path(res.path).name
    cv2.imwrite(str(out_path), plotted)
    saved += 1

print(f"Done. Processed {len(results)} images and saved {saved} overlays to: {overlay_dir}")

Done. Processed 120 images and saved 120 overlays to: C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Iphone\yolo_iphone_dataset\split\images\val_overlay


### 9. Run Inference on OOS Images and Save Overlays
Runs the trained model on out-of-sample images and writes visualization overlays to the OOS output folder.

In [12]:
weights_path = Path(r"C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Code\yolo11l_iphone\weights\best.pt")
val_dir = Path(r"C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\OOS images")
overlay_dir = Path(r"C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\OOS overlay")

# Load model and run prediction on validation images
model = YOLO(str(weights_path))
results = model.predict(source=str(val_dir), conf=0.25, iou=0.7, save=False, verbose=False)

# Save per-image overlays with bounding boxes/labels
saved = 0
for res in results:
    plotted = res.plot()
    out_path = overlay_dir / Path(res.path).name
    cv2.imwrite(str(out_path), plotted)
    saved += 1

print(f"Done. Processed {len(results)} images and saved {saved} overlays to: {overlay_dir}")

Done. Processed 25 images and saved 25 overlays to: C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\OOS overlay


### 10. Export One Overlay per Recyclable Class for Report
Selects the first sample from each SAM3 class output, redraws YOLO boxes from label files, and saves the overlays to the report folder.

In [24]:
REPORT_DIR = Path(r"C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Report use")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

class_id_to_name = {v: k for k, v in CLASS_NAME_TO_ID.items()}
class_colors = {
    0: (255, 0, 0),      # blue bins
    1: (0, 255, 255),    # metal can
    2: (0, 255, 0),      # newspaper
    3: (255, 0, 255),    # plastic bag
    4: (0, 165, 255),    # plastic bottle
}

sam3_bbox_dirs = {
    "metal can": DATA_ROOT / "Sam3 Metal can output" / "bbox",
    "newspaper": DATA_ROOT / "Sam3 Newspaper output" / "bbox",
    "plastic bag": DATA_ROOT / "Sam3 Plastic bag output" / "bbox",
    "plastic bottle": DATA_ROOT / "Sam3 Plastic bottle output" / "bbox",
}

def yolo_to_xyxy(xc, yc, w, h, img_w, img_h):
    x1 = int((xc - w / 2) * img_w)
    y1 = int((yc - h / 2) * img_h)
    x2 = int((xc + w / 2) * img_w)
    y2 = int((yc + h / 2) * img_h)
    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(0, min(x2, img_w - 1))
    y2 = max(0, min(y2, img_h - 1))
    return x1, y1, x2, y2

def find_original_image(class_name: str, stem: str) -> Path | None:
    src_dir = DATA_ROOT / class_name
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        candidate = src_dir / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None

saved_files = []
for class_name, bbox_dir in sam3_bbox_dirs.items():
    image_files = sorted(bbox_dir.glob("*_bbox.jpg"))
    if not image_files:
        print(f"{class_name}: no bbox images found")
        continue

    first_img_path = image_files[0]
    stem = first_img_path.stem.replace("_bbox", "")
    label_path = bbox_dir / f"{stem}.txt"

    # Save original image for copy-paste use in reports.
    original_path = find_original_image(class_name, stem)
    if original_path is not None:
        original_img = cv2.imread(str(original_path))
    else:
        original_img = cv2.imread(str(first_img_path))

    if original_img is not None:
        original_out_path = REPORT_DIR / f"report_{class_name.replace(' ', '_')}_original.jpg"
        cv2.imwrite(str(original_out_path), original_img)
        saved_files.append(original_out_path.name)
        print(f"{class_name}: saved original {original_out_path}")
    else:
        print(f"{class_name}: failed to read original image")

    img = cv2.imread(str(first_img_path))
    if img is None:
        print(f"{class_name}: failed to read image {first_img_path.name}")
        continue

    img_h, img_w = img.shape[:2]
    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            try:
                cid = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])
            except ValueError:
                continue

            x1, y1, x2, y2 = yolo_to_xyxy(xc, yc, w, h, img_w, img_h)
            color = class_colors.get(cid, (255, 255, 255))
            label = class_id_to_name.get(cid, f"class_{cid}").upper()

            # Draw thick box for report readability.
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 20)

            # Draw label inside the box with a filled background.
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 7.5
            text_thickness = 5
            pad = 14
            (tw, th), baseline = cv2.getTextSize(label, font, font_scale, text_thickness)

            text_x = max(x1 + pad, 0)
            text_y = min(max(y1 + th + pad, th + pad), img_h - baseline - 1)

            bg_x1 = max(text_x - pad, 0)
            bg_y1 = max(text_y - th - pad, 0)
            bg_x2 = min(text_x + tw + pad, img_w - 1)
            bg_y2 = min(text_y + baseline + pad, img_h - 1)

            cv2.rectangle(img, (bg_x1, bg_y1), (bg_x2, bg_y2), color, -1)
            text_color = (255, 255, 255) if cid == 0 else (0, 0, 0)
            cv2.putText(
                img,
                label,
                (text_x, text_y),
                font,
                font_scale,
                text_color,
                text_thickness,
                cv2.LINE_AA,
            )
    else:
        print(f"{class_name}: label file not found for {first_img_path.name}")

    out_path = REPORT_DIR / f"report_{class_name.replace(' ', '_')}_overlay.jpg"
    cv2.imwrite(str(out_path), img)
    saved_files.append(out_path.name)
    print(f"{class_name}: saved overlay {out_path}")

print("\nSaved files:")
for name in saved_files:
    print(f"- {name}")

metal can: saved original C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Report use\report_metal_can_original.jpg
metal can: saved overlay C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Report use\report_metal_can_overlay.jpg
newspaper: saved original C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Report use\report_newspaper_original.jpg
newspaper: saved overlay C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Report use\report_newspaper_overlay.jpg
plastic bag: saved original C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Report use\report_plastic_bag_original.jpg
plastic bag: saved overlay C:\Users\shenl\OneDrive\Documents\NUS\MS2\CS5224-Cloud Computing\Project\Machine Learning\Data\Report use\report_plastic_bag_overlay.jpg
plastic bottle: saved orig